In [28]:
from gurobipy import GRB, Model, quicksum
import gurobipy as gb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus
import random
import math
import scipy.stats as sp
import gurobipy as gp
from itertools import permutations
import ast

In [29]:

# Read the data
cost_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/cost.csv')
time_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/time.csv')

In [30]:
cost

114

In [34]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm, expon, uniform

# Read the data and print the data types
print("Reading CSV files...")
cost_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/cost.csv')
time_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/time.csv')

# Print data types and first few rows
print("\nCost DataFrame Info:")
print(cost_df.info())
print("\nFirst few rows of cost data:")
print(cost_df.head())

print("\nTime DataFrame Info:")
print(time_df.info())
print("\nFirst few rows of time data:")
print(time_df.head())

# Convert string columns to numeric
cost_df = cost_df.apply(pd.to_numeric, errors='coerce')
time_df = time_df.apply(pd.to_numeric, errors='coerce')

# Print the processed data
print("\nProcessed Cost Matrix:")
print(cost_df)
print("\nProcessed Time Matrix:")
print(time_df)

# Define parameters
customers = list(range(1, 8))  # Customers 1-7
PENALTY_COST = 2000  # Cost per unit of shortage
DISPOSAL_COST = 5    # Cost per unit of disposal
MAX_TIME = 724       # Maximum total conversion time
INITIAL_PRODUCTION_COST = 1000  # Cost per unit of initial production

# Create cost and time matrices with error checking
cost_matrix = {}
time_matrix = {}
for i in range(7):
    for j in range(7):
        if i != j:
            try:
                cost_value = float(cost_df.iloc[i, j+1])
                time_value = float(time_df.iloc[i, j+1])
                if pd.isna(cost_value) or pd.isna(time_value):
                    print(f"Warning: NaN value found at position ({i+1}, {j+1})")
                    continue
                cost_matrix[(i+1, j+1)] = cost_value
                time_matrix[(i+1, j+1)] = time_value
            except (ValueError, IndexError) as e:
                print(f"Error processing data at position ({i+1}, {j+1}): {e}")

# Print the matrices to verify
print("\nCost Matrix:")
for (i,j), cost in cost_matrix.items():
    print(f"Cost to convert from {i} to {j}: ${cost:,.2f}")

print("\nTime Matrix:")
for (i,j), time in time_matrix.items():
    print(f"Time to convert from {i} to {j}: {time:,.2f} hours")

# Verify we have all necessary data
if len(cost_matrix) != 42 or len(time_matrix) != 42:  # 7 customers, 6 possible conversions each
    print("Warning: Missing some conversion costs or times")
    print(f"Number of cost entries: {len(cost_matrix)}")
    print(f"Number of time entries: {len(time_matrix)}")

# Rest of the code remains the same...

Reading CSV files...

Cost DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  7 non-null      object
 1   Customer 1  7 non-null      int64 
 2   Customer 2  7 non-null      int64 
 3   Customer 3  7 non-null      int64 
 4   Customer 4  7 non-null      int64 
 5   Customer 5  7 non-null      int64 
 6   Customer 6  7 non-null      int64 
 7   Customer 7  7 non-null      int64 
dtypes: int64(7), object(1)
memory usage: 576.0+ bytes
None

First few rows of cost data:
   Unnamed: 0  Customer 1  Customer 2  Customer 3  Customer 4  Customer 5  \
0  Customer 1        1000         761         443         513         485   
1  Customer 2         352        1000         956         574         575   
2  Customer 3         666         584        1000         120         266   
3  Customer 4         341         445         439        10

In [37]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm, expon, uniform

# Read the data
cost_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/cost.csv', index_col=0)
time_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/time.csv', index_col=0)

# Define parameters
customers = list(range(1, 8))  # Customers 1-7
PENALTY_COST = 2000  # Cost per unit of shortage
DISPOSAL_COST = 5    # Cost per unit of disposal
MAX_TIME = 724       # Maximum total conversion time
INITIAL_PRODUCTION_COST = 1000  # Cost per unit of initial production

# Create cost and time matrices
cost_matrix = {}
time_matrix = {}
for i in range(7):
    for j in range(7):
        if i != j:  # Skip diagonal elements (self-conversions)
            cost_matrix[(i+1, j+1)] = float(cost_df.iloc[i, j])
            time_matrix[(i+1, j+1)] = float(time_df.iloc[i, j])

# Create the model
model = gp.Model("Dye_Conversion_Optimization")

# Decision variables
conversions = model.addVars(cost_matrix.keys(), name="conversions")
disposals = model.addVars(customers, name="disposals")
shortages = model.addVars(customers, name="shortages")
initial_production = model.addVars(customers, name="initial_production")

# Expected demand for each customer
expected_demand = {
    1: norm(10000, 2000).mean(),  # Customer 1: Normal distribution
    2: expon(scale=15000).mean(), # Customer 2: Exponential distribution
    3: uniform(5000, 15000).mean(), # Customer 3: Uniform distribution
    4: norm(12000, 3000).mean(),  # Customer 4: Normal distribution
    5: expon(scale=20000).mean(), # Customer 5: Exponential distribution
    6: uniform(8000, 20000).mean(), # Customer 6: Uniform distribution
    7: norm(15000, 4000).mean()   # Customer 7: Normal distribution
}

# Print expected demands for debugging
print("\nExpected Demands:")
for customer, demand in expected_demand.items():
    print(f"Customer {customer}: {demand:,.2f} units")

# Objective function
total_cost = (
    gp.quicksum(conversions[i,j] * cost_matrix[i,j] for i,j in cost_matrix.keys()) +
    gp.quicksum(disposals[i] * DISPOSAL_COST for i in customers) +
    gp.quicksum(shortages[i] * PENALTY_COST for i in customers) +
    gp.quicksum(initial_production[i] * INITIAL_PRODUCTION_COST for i in customers)
)
model.setObjective(total_cost, GRB.MINIMIZE)

# Time constraint
model.addConstr(
    gp.quicksum(conversions[i,j] * time_matrix[i,j] for i,j in time_matrix.keys()) <= MAX_TIME,
    "time_limit"
)

# Balance constraints for each customer
for i in customers:
    model.addConstr(
        initial_production[i] + 
        gp.quicksum(conversions[j,i] for j in range(1,8) if (j,i) in conversions) -
        gp.quicksum(conversions[i,j] for j in range(1,8) if (i,j) in conversions) -
        disposals[i] + shortages[i] == expected_demand[i],
        f"balance_{i}"
    )

# Initial production limits - relaxed to 95% of expected demand
for i in customers:
    model.addConstr(initial_production[i] <= 0.95 * expected_demand[i], f"prod_limit_{i}")

# Minimum conversion requirement - reduced to 300 units
model.addConstr(
    gp.quicksum(conversions[i,j] for i,j in conversions.keys()) >= 300,
    "min_conversion"
)

# Set model parameters for better debugging
model.setParam('InfUnbdInfo', 1)
model.setParam('OutputFlag', 1)

# Optimize
try:
    model.optimize()
    
    # Print results
    print("\nOptimization Results:")
    print(f"Status: {model.status}")
    
    if model.status == GRB.OPTIMAL:
        print(f"Total Cost: ${model.objVal:,.2f}")
        
        print("\nInitial Production:")
        for i in customers:
            print(f"Customer {i}: {initial_production[i].X:,.2f} units")
        
        print("\nConversions:")
        for i,j in conversions.keys():
            if conversions[i,j].X > 0:
                print(f"From {i} to {j}: {conversions[i,j].X:,.2f} units")
        
        print("\nDisposals:")
        for i in customers:
            if disposals[i].X > 0:
                print(f"Customer {i}: {disposals[i].X:,.2f} units")
        
        print("\nShortages:")
        for i in customers:
            if shortages[i].X > 0:
                print(f"Customer {i}: {shortages[i].X:,.2f} units")
        
        print("\nTime Usage:")
        total_time = sum(conversions[i,j].X * time_matrix[i,j] for i,j in time_matrix.keys())
        print(f"Total conversion time used: {total_time:,.2f} hours")
    
    elif model.status == GRB.INFEASIBLE:
        print("Model is infeasible!")
        model.computeIIS()
        if model.IISMinimal:
            print("IIS is minimal\n")
        else:
            print("IIS is not minimal\n")
        print("\nConstraints in the IIS:")
        for c in model.getConstrs():
            if c.IISConstr:
                print(f"- {c.ConstrName}")
    
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded!")
        
    else:
        print(f"Optimization failed with status code: {model.status}")

except gp.GurobiError as e:
    print(f"Error code {e.errno}: {e}")


Expected Demands:
Customer 1: 10,000.00 units
Customer 2: 15,000.00 units
Customer 3: 12,500.00 units
Customer 4: 12,000.00 units
Customer 5: 20,000.00 units
Customer 6: 18,000.00 units
Customer 7: 15,000.00 units
Set parameter InfUnbdInfo to value 1
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
InfUnbdInfo  1

Optimize a model with 16 rows, 63 columns and 196 nonzeros
Model fingerprint: 0x2416c311
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [5e+00, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+02, 2e+04]
Presolve removed 13 rows and 27 columns
Presolve time: 0.00s
Presolved: 3 rows, 36 columns, 79 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.2117907e+07   1.925132e+03   0.000000e+00      0s
       5

e

In [38]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm, expon, uniform
import time

def generate_demand_scenario():
    """Generate a single scenario of demands for all customers"""
    return {
        1: norm(10000, 2000).rvs(),    # Customer 1: Normal distribution
        2: expon(scale=15000).rvs(),   # Customer 2: Exponential distribution
        3: uniform(5000, 15000).rvs(), # Customer 3: Uniform distribution
        4: norm(12000, 3000).rvs(),    # Customer 4: Normal distribution
        5: expon(scale=20000).rvs(),   # Customer 5: Exponential distribution
        6: uniform(8000, 20000).rvs(), # Customer 6: Uniform distribution
        7: norm(15000, 4000).rvs()     # Customer 7: Normal distribution
    }

def solve_optimization(demands):
    """Solve the optimization problem for a given demand scenario"""
    # Read the data
    cost_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/cost.csv', index_col=0)
    time_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/time.csv', index_col=0)

    # Define parameters
    customers = list(range(1, 8))
    PENALTY_COST = 2000
    DISPOSAL_COST = 5
    MAX_TIME = 724
    INITIAL_PRODUCTION_COST = 1000

    # Create cost and time matrices
    cost_matrix = {}
    time_matrix = {}
    for i in range(7):
        for j in range(7):
            if i != j:
                cost_matrix[(i+1, j+1)] = float(cost_df.iloc[i, j])
                time_matrix[(i+1, j+1)] = float(time_df.iloc[i, j])

    # Create the model
    model = gp.Model("Dye_Conversion_Optimization")

    # Decision variables
    conversions = model.addVars(cost_matrix.keys(), name="conversions")
    disposals = model.addVars(customers, name="disposals")
    shortages = model.addVars(customers, name="shortages")
    initial_production = model.addVars(customers, name="initial_production")

    # Objective function
    total_cost = (
        gp.quicksum(conversions[i,j] * cost_matrix[i,j] for i,j in cost_matrix.keys()) +
        gp.quicksum(disposals[i] * DISPOSAL_COST for i in customers) +
        gp.quicksum(shortages[i] * PENALTY_COST for i in customers) +
        gp.quicksum(initial_production[i] * INITIAL_PRODUCTION_COST for i in customers)
    )
    model.setObjective(total_cost, GRB.MINIMIZE)

    # Time constraint
    model.addConstr(
        gp.quicksum(conversions[i,j] * time_matrix[i,j] for i,j in time_matrix.keys()) <= MAX_TIME,
        "time_limit"
    )

    # Balance constraints for each customer
    for i in customers:
        model.addConstr(
            initial_production[i] + 
            gp.quicksum(conversions[j,i] for j in range(1,8) if (j,i) in conversions) -
            gp.quicksum(conversions[i,j] for j in range(1,8) if (i,j) in conversions) -
            disposals[i] + shortages[i] == demands[i],
            f"balance_{i}"
        )

    # Initial production limits
    for i in customers:
        model.addConstr(initial_production[i] <= 0.95 * demands[i], f"prod_limit_{i}")

    # Minimum conversion requirement
    model.addConstr(
        gp.quicksum(conversions[i,j] for i,j in conversions.keys()) >= 300,
        "min_conversion"
    )

    # Optimize
    model.optimize()

    if model.status == GRB.OPTIMAL:
        return model.objVal
    else:
        return None

def run_monte_carlo_simulation(num_trials=50, scenarios_per_trial=100):
    """Run Monte Carlo simulation with SAA"""
    all_costs = []
    
    for trial in range(num_trials):
        print(f"\nRunning trial {trial + 1}/{num_trials}")
        trial_costs = []
        
        for scenario in range(scenarios_per_trial):
            demands = generate_demand_scenario()
            cost = solve_optimization(demands)
            if cost is not None:
                trial_costs.append(cost)
        
        if trial_costs:
            avg_cost = np.mean(trial_costs)
            all_costs.append(avg_cost)
            print(f"Trial {trial + 1} average cost: ${avg_cost:,.2f}")
    
    return all_costs

def calculate_confidence_interval(costs, confidence=0.95):
    """Calculate confidence interval for the costs"""
    mean_cost = np.mean(costs)
    std_error = np.std(costs) / np.sqrt(len(costs))
    z_score = norm.ppf((1 + confidence) / 2)
    margin_of_error = z_score * std_error
    
    return mean_cost, (mean_cost - margin_of_error, mean_cost + margin_of_error)

if __name__ == "__main__":
    print("Starting Monte Carlo simulation...")
    start_time = time.time()
    
    costs = run_monte_carlo_simulation()
    mean_cost, ci = calculate_confidence_interval(costs)
    
    print("\nFinal Results:")
    print(f"Number of successful trials: {len(costs)}")
    print(f"Optimal Expected Cost: ${mean_cost:,.2f}")
    print(f"95% Confidence Interval: [${ci[0]:,.2f}, ${ci[1]:,.2f}]")
    print(import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import networkx as nx

# -----------------------
# Data Extraction and Preprocessing
# -----------------------
df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/delivery.csv')

# Define customers 1–15 and vans 1–3
customers = list(range(1, 16))
vans = list(range(1, 4))
weights = df['Size'].values
depot_distances = df['Depot'].values

# Create a dictionary for customer-to-customer distances.
# Assumes that columns 4 onward (0-indexed: column index 3+) of the CSV hold inter-customer distances.
distances = {}
for i in range(15):
    for j in range(15):
        if i != j:
            distances[(i+1, j+1)] = df.iloc[i, j+3]

# -----------------------
# Subtour Detection Function
# -----------------------
def find_subtours(edges):
    """
    Given a list of edges (tuples (i,j)), this function returns a list of subtours.
    A subtour is defined as a connected component of the route graph with fewer nodes than the set of
    customers assigned to that vehicle.
    """
    G = nx.Graph()
    G.add_edges_from(edges)
    subtours = []
    # Each connected component is a potential tour.
    for comp in nx.connected_components(G):
        # In our formulation, a full tour should visit all customers assigned to the van.
        # If the component is smaller, then it is a subtour.
        if len(comp) < len(G.nodes()):
            subtours.append(list(comp))
    return subtours

# -----------------------
# Callback for Lazy Subtour Elimination
# -----------------------
# Global counter to track how many lazy constraints are added.
lazy_counter = {"count": 0}

def mycallback(model, where):
    if where == GRB.Callback.MIPSOL:
        # For each van, examine the current solution.
        for k in vans:
            # Collect edges for vehicle k with x > 0.5 in the current solution.
            selected_edges = []
            for i in customers:
                for j in customers:
                    if i != j:
                        sol_val = model.cbGetSolution(x[i, j, k])
                        if sol_val > 0.5:
                            selected_edges.append((i, j))
            subtours = find_subtours(selected_edges)
            for subtour in subtours:
                # Add a lazy constraint: For subtour S, the sum of arcs among nodes in S must be <= |S|-1.
                model.cbLazy(gp.quicksum(x[i, j, k] for i in subtour for j in subtour if i != j) <= len(subtour) - 1)
                lazy_counter["count"] += 1
                print(f"Lazy constraint added for van {k} subtour: {subtour}")

# -----------------------
# Build the MILP Model
# -----------------------
model = gp.Model("FedEx_Routing_With_Callback")

# Decision Variables
# x[i,j,k] = 1 if van k travels directly from customer i to customer j.
x = model.addVars([(i, j, k) for i in customers for j in customers for k in vans if i != j],
                  vtype=GRB.BINARY, name="route")
# y[i,k] = 1 if customer i is assigned to van k.
y = model.addVars([(i, k) for i in customers for k in vans],
                  vtype=GRB.BINARY, name="assignment")
# z[k] = total weight delivered by van k.
z = model.addVars(vans, lb=0, name="weight")

# Auxiliary variables to capture maximum and minimum load across vans.
max_weight = model.addVar(lb=0, name="max_weight")
min_weight = model.addVar(lb=0, name="min_weight")

# -----------------------
# Objective: Minimize difference in loads
# -----------------------
model.setObjective(max_weight - min_weight, GRB.MINIMIZE)

# -----------------------
# Constraints
# -----------------------

# (1) Each customer is assigned to exactly one van.
for i in customers:
    model.addConstr(gp.quicksum(y[i, k] for k in vans) == 1, name=f"assign_{i}")

# (2) All vans must serve at least one customer.
for k in vans:
    model.addConstr(gp.quicksum(y[i, k] for i in customers) >= 1, name=f"van_usage_{k}")

# (3) Weight (load) constraints for each van.
for k in vans:
    model.addConstr(z[k] == gp.quicksum(weights[i-1] * y[i, k] for i in customers), name=f"weight_calc_{k}")
    model.addConstr(z[k] <= 15000, name=f"capacity_{k}")
    model.addConstr(max_weight >= z[k], name=f"max_bound_{k}")
    model.addConstr(min_weight <= z[k], name=f"min_bound_{k}")

# (4) Distance constraints for each van:
for k in vans:
    model.addConstr(
        gp.quicksum(depot_distances[i-1] * y[i, k] for i in customers) +
        gp.quicksum(distances[(i, j)] * x[i, j, k] for i in customers for j in customers if i != j)
        <= 254,
        name=f"range_limit_{k}"
    )

# (5) Flow conservation: if customer i is served by van k, then the number of departures equals the assignment.
for k in vans:
    for i in customers:
        model.addConstr(gp.quicksum(x[i, j, k] for j in customers if i != j) == y[i, k],
                        name=f"flow_out_{i}_{k}")
        model.addConstr(gp.quicksum(x[j, i, k] for j in customers if i != j) == y[i, k],
                        name=f"flow_in_{i}_{k}")

# (6) Special operational constraints:
# - No more than 2 of customers 7, 8, 9 on the same van.
for k in vans:
    model.addConstr(gp.quicksum(y[i, k] for i in [7, 8, 9]) <= 2, name=f"special_7_9_{k}")

# - Customers 10, 11, 12 must be assigned to the same van.
for k in vans:
    model.addConstr(y[10, k] == y[11, k], name=f"same_van_10_11_{k}")
    model.addConstr(y[11, k] == y[12, k], name=f"same_van_11_12_{k}")

# - If customer 1 is assigned, at least one of 13 or 14 must be on that van.
for k in vans:
    model.addConstr(y[1, k] <= y[13, k] + y[14, k], name=f"cust1_req_{k}")

# - Customer 2 cannot be with customers 3, 4, 5.
for k in vans:
    model.addConstr(y[2, k] + y[3, k] <= 1, name=f"cust2_3_{k}")
    model.addConstr(y[2, k] + y[4, k] <= 1, name=f"cust2_4_{k}")
    model.addConstr(y[2, k] + y[5, k] <= 1, name=f"cust2_5_{k}")

# - Maximum 5 deliveries per van.
for k in vans:
    model.addConstr(gp.quicksum(y[i, k] for i in customers) <= 5, name=f"max_deliv_{k}")

# -----------------------
# Set up for Lazy Constraints
# -----------------------
model.Params.LazyConstraints = 1

# -----------------------
# Optimize using the Callback
# -----------------------
model.optimize(mycallback)

# -----------------------
# Report Results
# -----------------------
print("\n=== Optimization Results (with Callback Lazy Constraints) ===")
print(f"Status: {model.status}")
print(f"Optimal Objective Value (Max Weight Difference): {model.objVal:.2f} lbs")
print(f"Total Number of Lazy Constraints Added: {lazy_counter['count']}")
f"Total simulation time: {time.time() - start_time:.2f} seconds")

Starting Monte Carlo simulation...

Running trial 1/50
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 63 columns and 196 nonzeros
Model fingerprint: 0x2bd553dc
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [5e+00, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+02, 2e+04]
Presolve removed 11 rows and 15 columns
Presolve time: 0.01s
Presolved: 5 rows, 48 columns, 121 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.5768344e+07   1.392859e+03   0.000000e+00      0s
       9    6.7544915e+07   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.01 seconds (0.00 work units)
Optimal objective  6.754491529e+07
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 p

In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm, expon, uniform

# Read the data
cost_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/cost.csv', index_col=0)
time_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/time.csv', index_col=0)

# Define parameters
customers = list(range(1, 8))  # Customers 1-7
PENALTY_COST = 2000  # Cost per unit of shortage
DISPOSAL_COST = 5    # Cost per unit of disposal
MAX_TIME = 724       # Maximum total conversion time
INITIAL_PRODUCTION_COST = 1000  # Cost per unit of initial production

# Create cost and time matrices
cost_matrix = {}
time_matrix = {}
for i in range(7):
    for j in range(7):
        if i != j:  # Skip diagonal elements (self-conversions)
            cost_matrix[(i+1, j+1)] = float(cost_df.iloc[i, j])
            time_matrix[(i+1, j+1)] = float(time_df.iloc[i, j])

# Create the model
model = gp.Model("Dye_Conversion_Optimization")

# Decision variables
conversions = model.addVars(cost_matrix.keys(), name="conversions")
disposals = model.addVars(customers, name="disposals")
shortages = model.addVars(customers, name="shortages")
initial_production = model.addVars(customers, name="initial_production")

# Expected demand for each customer
expected_demand = {
    1: norm(10000, 2000).mean(),  # Customer 1: Normal distribution
    2: expon(scale=15000).mean(), # Customer 2: Exponential distribution
    3: uniform(5000, 15000).mean(), # Customer 3: Uniform distribution
    4: norm(12000, 3000).mean(),  # Customer 4: Normal distribution
    5: expon(scale=20000).mean(), # Customer 5: Exponential distribution
    6: uniform(8000, 20000).mean(), # Customer 6: Uniform distribution
    7: norm(15000, 4000).mean()   # Customer 7: Normal distribution
}

# Print expected demands for debugging
print("\nExpected Demands:")
for customer, demand in expected_demand.items():
    print(f"Customer {customer}: {demand:,.2f} units")

# Objective function
total_cost = (
    gp.quicksum(conversions[i,j] * cost_matrix[i,j] for i,j in cost_matrix.keys()) +
    gp.quicksum(disposals[i] * DISPOSAL_COST for i in customers) +
    gp.quicksum(shortages[i] * PENALTY_COST for i in customers) +
    gp.quicksum(initial_production[i] * INITIAL_PRODUCTION_COST for i in customers)
)
model.setObjective(total_cost, GRB.MINIMIZE)

# Time constraint
model.addConstr(
    gp.quicksum(conversions[i,j] * time_matrix[i,j] for i,j in time_matrix.keys()) <= MAX_TIME,
    "time_limit"
)

# Balance constraints for each customer
for i in customers:
    model.addConstr(
        initial_production[i] + 
        gp.quicksum(conversions[j,i] for j in range(1,8) if (j,i) in conversions) -
        gp.quicksum(conversions[i,j] for j in range(1,8) if (i,j) in conversions) -
        disposals[i] + shortages[i] == expected_demand[i],
        f"balance_{i}"
    )

# Initial production limits - relaxed to 95% of expected demand
for i in customers:
    model.addConstr(initial_production[i] <= 0.95 * expected_demand[i], f"prod_limit_{i}")

# Minimum conversion requirement - reduced to 300 units
model.addConstr(
    gp.quicksum(conversions[i,j] for i,j in conversions.keys()) >= 300,
    "min_conversion"
)

# Set model parameters for better debugging
model.setParam('InfUnbdInfo', 1)
model.setParam('OutputFlag', 1)

# Optimize
try:
    model.optimize()
    
    # Print results
    print("\nOptimization Results:")
    print(f"Status: {model.status}")
    
    if model.status == GRB.OPTIMAL:
        print(f"Total Cost: ${model.objVal:,.2f}")
        
        print("\nInitial Production:")
        for i in customers:
            print(f"Customer {i}: {initial_production[i].X:,.2f} units")
        
        print("\nConversions:")
        for i,j in conversions.keys():
            if conversions[i,j].X > 0:
                print(f"From {i} to {j}: {conversions[i,j].X:,.2f} units")
        
        print("\nDisposals:")
        for i in customers:
            if disposals[i].X > 0:
                print(f"Customer {i}: {disposals[i].X:,.2f} units")
        
        print("\nShortages:")
        for i in customers:
            if shortages[i].X > 0:
                print(f"Customer {i}: {shortages[i].X:,.2f} units")
        
        print("\nTime Usage:")
        total_time = sum(conversions[i,j].X * time_matrix[i,j] for i,j in time_matrix.keys())
        print(f"Total conversion time used: {total_time:,.2f} hours")
    
    elif model.status == GRB.INFEASIBLE:
        print("Model is infeasible!")
        model.computeIIS()
        if model.IISMinimal:
            print("IIS is minimal\n")
        else:
            print("IIS is not minimal\n")
        print("\nConstraints in the IIS:")
        for c in model.getConstrs():
            if c.IISConstr:
                print(f"- {c.ConstrName}")
    
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded!")
        
    else:
        print(f"Optimization failed with status code: {model.status}")

except gp.GurobiError as e:
    print(f"Error code {e.errno}: {e}")


Expected Demands:
Customer 1: 10,000.00 units
Customer 2: 15,000.00 units
Customer 3: 12,500.00 units
Customer 4: 12,000.00 units
Customer 5: 20,000.00 units
Customer 6: 18,000.00 units
Customer 7: 15,000.00 units
Set parameter InfUnbdInfo to value 1
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
InfUnbdInfo  1

Optimize a model with 16 rows, 63 columns and 196 nonzeros
Model fingerprint: 0x2416c311
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [5e+00, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+02, 2e+04]
Presolve removed 13 rows and 27 columns
Presolve time: 0.00s
Presolved: 3 rows, 36 columns, 79 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.2117907e+07   1.925132e+03   0.000000e+00      0s
       5

In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm, expon, uniform

# Read the data
cost_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/cost.csv', index_col=0)
time_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/time.csv', index_col=0)

# Define parameters
customers = list(range(1, 8))  # Customers 1-7
PENALTY_COST = 2000  # Cost per unit of shortage
DISPOSAL_COST = 5    # Cost per unit of disposal
MAX_TIME = 724       # Maximum total conversion time
INITIAL_PRODUCTION_COST = 1000  # Cost per unit of initial production

# Create cost and time matrices
cost_matrix = {}
time_matrix = {}
for i in range(7):
    for j in range(7):
        if i != j:  # Skip diagonal elements (self-conversions)
            cost_matrix[(i+1, j+1)] = float(cost_df.iloc[i, j])
            time_matrix[(i+1, j+1)] = float(time_df.iloc[i, j])

# Create the model
model = gp.Model("Dye_Conversion_Optimization")

# Decision variables
conversions = model.addVars(cost_matrix.keys(), name="conversions")
disposals = model.addVars(customers, name="disposals")
shortages = model.addVars(customers, name="shortages")
initial_production = model.addVars(customers, name="initial_production")

# Expected demand for each customer
expected_demand = {
    1: norm(10000, 2000).mean(),  # Customer 1: Normal distribution
    2: expon(scale=15000).mean(), # Customer 2: Exponential distribution
    3: uniform(5000, 15000).mean(), # Customer 3: Uniform distribution
    4: norm(12000, 3000).mean(),  # Customer 4: Normal distribution
    5: expon(scale=20000).mean(), # Customer 5: Exponential distribution
    6: uniform(8000, 20000).mean(), # Customer 6: Uniform distribution
    7: norm(15000, 4000).mean()   # Customer 7: Normal distribution
}

# Print expected demands for debugging
print("\nExpected Demands:")
for customer, demand in expected_demand.items():
    print(f"Customer {customer}: {demand:,.2f} units")

# Objective function
total_cost = (
    gp.quicksum(conversions[i,j] * cost_matrix[i,j] for i,j in cost_matrix.keys()) +
    gp.quicksum(disposals[i] * DISPOSAL_COST for i in customers) +
    gp.quicksum(shortages[i] * PENALTY_COST for i in customers) +
    gp.quicksum(initial_production[i] * INITIAL_PRODUCTION_COST for i in customers)
)
model.setObjective(total_cost, GRB.MINIMIZE)

# Time constraint
model.addConstr(
    gp.quicksum(conversions[i,j] * time_matrix[i,j] for i,j in time_matrix.keys()) <= MAX_TIME,
    "time_limit"
)

# Balance constraints for each customer
for i in customers:
    model.addConstr(
        initial_production[i] + 
        gp.quicksum(conversions[j,i] for j in range(1,8) if (j,i) in conversions) -
        gp.quicksum(conversions[i,j] for j in range(1,8) if (i,j) in conversions) -
        disposals[i] + shortages[i] == expected_demand[i],
        f"balance_{i}"
    )

# Initial production limits - relaxed to 95% of expected demand
for i in customers:
    model.addConstr(initial_production[i] <= 0.95 * expected_demand[i], f"prod_limit_{i}")

# Minimum conversion requirement - reduced to 300 units
model.addConstr(
    gp.quicksum(conversions[i,j] for i,j in conversions.keys()) >= 300,
    "min_conversion"
)

# Set model parameters for better debugging
model.setParam('InfUnbdInfo', 1)
model.setParam('OutputFlag', 1)

# Optimize
try:
    model.optimize()
    
    # Print results
    print("\nOptimization Results:")
    print(f"Status: {model.status}")
    
    if model.status == GRB.OPTIMAL:
        print(f"Total Cost: ${model.objVal:,.2f}")
        
        print("\nInitial Production:")
        for i in customers:
            print(f"Customer {i}: {initial_production[i].X:,.2f} units")
        
        print("\nConversions:")
        for i,j in conversions.keys():
            if conversions[i,j].X > 0:
                print(f"From {i} to {j}: {conversions[i,j].X:,.2f} units")
        
        print("\nDisposals:")
        for i in customers:
            if disposals[i].X > 0:
                print(f"Customer {i}: {disposals[i].X:,.2f} units")
        
        print("\nShortages:")
        for i in customers:
            if shortages[i].X > 0:
                print(f"Customer {i}: {shortages[i].X:,.2f} units")
        
        print("\nTime Usage:")
        total_time = sum(conversions[i,j].X * time_matrix[i,j] for i,j in time_matrix.keys())
        print(f"Total conversion time used: {total_time:,.2f} hours")
    
    elif model.status == GRB.INFEASIBLE:
        print("Model is infeasible!")
        model.computeIIS()
        if model.IISMinimal:
            print("IIS is minimal\n")
        else:
            print("IIS is not minimal\n")
        print("\nConstraints in the IIS:")
        for c in model.getConstrs():
            if c.IISConstr:
                print(f"- {c.ConstrName}")
    
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded!")
        
    else:
        print(f"Optimization failed with status code: {model.status}")

except gp.GurobiError as e:
    print(f"Error code {e.errno}: {e}")


Expected Demands:
Customer 1: 10,000.00 units
Customer 2: 15,000.00 units
Customer 3: 12,500.00 units
Customer 4: 12,000.00 units
Customer 5: 20,000.00 units
Customer 6: 18,000.00 units
Customer 7: 15,000.00 units
Set parameter InfUnbdInfo to value 1
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
InfUnbdInfo  1

Optimize a model with 16 rows, 63 columns and 196 nonzeros
Model fingerprint: 0x2416c311
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [5e+00, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+02, 2e+04]
Presolve removed 13 rows and 27 columns
Presolve time: 0.00s
Presolved: 3 rows, 36 columns, 79 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.2117907e+07   1.925132e+03   0.000000e+00      0s
       5

In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm, expon, uniform

# Read the data
cost_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/cost.csv', index_col=0)
time_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/time.csv', index_col=0)

# Define parameters
customers = list(range(1, 8))  # Customers 1-7
PENALTY_COST = 2000  # Cost per unit of shortage
DISPOSAL_COST = 5    # Cost per unit of disposal
MAX_TIME = 724       # Maximum total conversion time
INITIAL_PRODUCTION_COST = 1000  # Cost per unit of initial production

# Create cost and time matrices
cost_matrix = {}
time_matrix = {}
for i in range(7):
    for j in range(7):
        if i != j:  # Skip diagonal elements (self-conversions)
            cost_matrix[(i+1, j+1)] = float(cost_df.iloc[i, j])
            time_matrix[(i+1, j+1)] = float(time_df.iloc[i, j])

# Create the model
model = gp.Model("Dye_Conversion_Optimization")

# Decision variables
conversions = model.addVars(cost_matrix.keys(), name="conversions")
disposals = model.addVars(customers, name="disposals")
shortages = model.addVars(customers, name="shortages")
initial_production = model.addVars(customers, name="initial_production")

# Expected demand for each customer
expected_demand = {
    1: norm(10000, 2000).mean(),  # Customer 1: Normal distribution
    2: expon(scale=15000).mean(), # Customer 2: Exponential distribution
    3: uniform(5000, 15000).mean(), # Customer 3: Uniform distribution
    4: norm(12000, 3000).mean(),  # Customer 4: Normal distribution
    5: expon(scale=20000).mean(), # Customer 5: Exponential distribution
    6: uniform(8000, 20000).mean(), # Customer 6: Uniform distribution
    7: norm(15000, 4000).mean()   # Customer 7: Normal distribution
}

# Print expected demands for debugging
print("\nExpected Demands:")
for customer, demand in expected_demand.items():
    print(f"Customer {customer}: {demand:,.2f} units")

# Objective function
total_cost = (
    gp.quicksum(conversions[i,j] * cost_matrix[i,j] for i,j in cost_matrix.keys()) +
    gp.quicksum(disposals[i] * DISPOSAL_COST for i in customers) +
    gp.quicksum(shortages[i] * PENALTY_COST for i in customers) +
    gp.quicksum(initial_production[i] * INITIAL_PRODUCTION_COST for i in customers)
)
model.setObjective(total_cost, GRB.MINIMIZE)

# Time constraint
model.addConstr(
    gp.quicksum(conversions[i,j] * time_matrix[i,j] for i,j in time_matrix.keys()) <= MAX_TIME,
    "time_limit"
)

# Balance constraints for each customer
for i in customers:
    model.addConstr(
        initial_production[i] + 
        gp.quicksum(conversions[j,i] for j in range(1,8) if (j,i) in conversions) -
        gp.quicksum(conversions[i,j] for j in range(1,8) if (i,j) in conversions) -
        disposals[i] + shortages[i] == expected_demand[i],
        f"balance_{i}"
    )

# Initial production limits - relaxed to 95% of expected demand
for i in customers:
    model.addConstr(initial_production[i] <= 0.95 * expected_demand[i], f"prod_limit_{i}")

# Minimum conversion requirement - reduced to 300 units
model.addConstr(
    gp.quicksum(conversions[i,j] for i,j in conversions.keys()) >= 300,
    "min_conversion"
)

# Set model parameters for better debugging
model.setParam('InfUnbdInfo', 1)
model.setParam('OutputFlag', 1)

# Optimize
try:
    model.optimize()
    
    # Print results
    print("\nOptimization Results:")
    print(f"Status: {model.status}")
    
    if model.status == GRB.OPTIMAL:
        print(f"Total Cost: ${model.objVal:,.2f}")
        
        print("\nInitial Production:")
        for i in customers:
            print(f"Customer {i}: {initial_production[i].X:,.2f} units")
        
        print("\nConversions:")
        for i,j in conversions.keys():
            if conversions[i,j].X > 0:
                print(f"From {i} to {j}: {conversions[i,j].X:,.2f} units")
        
        print("\nDisposals:")
        for i in customers:
            if disposals[i].X > 0:
                print(f"Customer {i}: {disposals[i].X:,.2f} units")
        
        print("\nShortages:")
        for i in customers:
            if shortages[i].X > 0:
                print(f"Customer {i}: {shortages[i].X:,.2f} units")
        
        print("\nTime Usage:")
        total_time = sum(conversions[i,j].X * time_matrix[i,j] for i,j in time_matrix.keys())
        print(f"Total conversion time used: {total_time:,.2f} hours")
    
    elif model.status == GRB.INFEASIBLE:
        print("Model is infeasible!")
        model.computeIIS()
        if model.IISMinimal:
            print("IIS is minimal\n")
        else:
            print("IIS is not minimal\n")
        print("\nConstraints in the IIS:")
        for c in model.getConstrs():
            if c.IISConstr:
                print(f"- {c.ConstrName}")
    
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded!")
        
    else:
        print(f"Optimization failed with status code: {model.status}")

except gp.GurobiError as e:
    print(f"Error code {e.errno}: {e}")


Expected Demands:
Customer 1: 10,000.00 units
Customer 2: 15,000.00 units
Customer 3: 12,500.00 units
Customer 4: 12,000.00 units
Customer 5: 20,000.00 units
Customer 6: 18,000.00 units
Customer 7: 15,000.00 units
Set parameter InfUnbdInfo to value 1
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
InfUnbdInfo  1

Optimize a model with 16 rows, 63 columns and 196 nonzeros
Model fingerprint: 0x2416c311
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [5e+00, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+02, 2e+04]
Presolve removed 13 rows and 27 columns
Presolve time: 0.00s
Presolved: 3 rows, 36 columns, 79 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.2117907e+07   1.925132e+03   0.000000e+00      0s
       5

In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm, expon, uniform

# Read the data
cost_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/cost.csv', index_col=0)
time_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/time.csv', index_col=0)

# Define parameters
customers = list(range(1, 8))  # Customers 1-7
PENALTY_COST = 2000  # Cost per unit of shortage
DISPOSAL_COST = 5    # Cost per unit of disposal
MAX_TIME = 724       # Maximum total conversion time
INITIAL_PRODUCTION_COST = 1000  # Cost per unit of initial production

# Create cost and time matrices
cost_matrix = {}
time_matrix = {}
for i in range(7):
    for j in range(7):
        if i != j:  # Skip diagonal elements (self-conversions)
            cost_matrix[(i+1, j+1)] = float(cost_df.iloc[i, j])
            time_matrix[(i+1, j+1)] = float(time_df.iloc[i, j])

# Create the model
model = gp.Model("Dye_Conversion_Optimization")

# Decision variables
conversions = model.addVars(cost_matrix.keys(), name="conversions")
disposals = model.addVars(customers, name="disposals")
shortages = model.addVars(customers, name="shortages")
initial_production = model.addVars(customers, name="initial_production")

# Expected demand for each customer
expected_demand = {
    1: norm(10000, 2000).mean(),  # Customer 1: Normal distribution
    2: expon(scale=15000).mean(), # Customer 2: Exponential distribution
    3: uniform(5000, 15000).mean(), # Customer 3: Uniform distribution
    4: norm(12000, 3000).mean(),  # Customer 4: Normal distribution
    5: expon(scale=20000).mean(), # Customer 5: Exponential distribution
    6: uniform(8000, 20000).mean(), # Customer 6: Uniform distribution
    7: norm(15000, 4000).mean()   # Customer 7: Normal distribution
}

# Print expected demands for debugging
print("\nExpected Demands:")
for customer, demand in expected_demand.items():
    print(f"Customer {customer}: {demand:,.2f} units")

# Objective function
total_cost = (
    gp.quicksum(conversions[i,j] * cost_matrix[i,j] for i,j in cost_matrix.keys()) +
    gp.quicksum(disposals[i] * DISPOSAL_COST for i in customers) +
    gp.quicksum(shortages[i] * PENALTY_COST for i in customers) +
    gp.quicksum(initial_production[i] * INITIAL_PRODUCTION_COST for i in customers)
)
model.setObjective(total_cost, GRB.MINIMIZE)

# Time constraint
model.addConstr(
    gp.quicksum(conversions[i,j] * time_matrix[i,j] for i,j in time_matrix.keys()) <= MAX_TIME,
    "time_limit"
)

# Balance constraints for each customer
for i in customers:
    model.addConstr(
        initial_production[i] + 
        gp.quicksum(conversions[j,i] for j in range(1,8) if (j,i) in conversions) -
        gp.quicksum(conversions[i,j] for j in range(1,8) if (i,j) in conversions) -
        disposals[i] + shortages[i] == expected_demand[i],
        f"balance_{i}"
    )

# Initial production limits - relaxed to 95% of expected demand
for i in customers:
    model.addConstr(initial_production[i] <= 0.95 * expected_demand[i], f"prod_limit_{i}")

# Minimum conversion requirement - reduced to 300 units
model.addConstr(
    gp.quicksum(conversions[i,j] for i,j in conversions.keys()) >= 300,
    "min_conversion"
)

# Set model parameters for better debugging
model.setParam('InfUnbdInfo', 1)
model.setParam('OutputFlag', 1)

# Optimize
try:
    model.optimize()
    
    # Print results
    print("\nOptimization Results:")
    print(f"Status: {model.status}")
    
    if model.status == GRB.OPTIMAL:
        print(f"Total Cost: ${model.objVal:,.2f}")
        
        print("\nInitial Production:")
        for i in customers:
            print(f"Customer {i}: {initial_production[i].X:,.2f} units")
        
        print("\nConversions:")
        for i,j in conversions.keys():
            if conversions[i,j].X > 0:
                print(f"From {i} to {j}: {conversions[i,j].X:,.2f} units")
        
        print("\nDisposals:")
        for i in customers:
            if disposals[i].X > 0:
                print(f"Customer {i}: {disposals[i].X:,.2f} units")
        
        print("\nShortages:")
        for i in customers:
            if shortages[i].X > 0:
                print(f"Customer {i}: {shortages[i].X:,.2f} units")
        
        print("\nTime Usage:")
        total_time = sum(conversions[i,j].X * time_matrix[i,j] for i,j in time_matrix.keys())
        print(f"Total conversion time used: {total_time:,.2f} hours")
    
    elif model.status == GRB.INFEASIBLE:
        print("Model is infeasible!")
        model.computeIIS()
        if model.IISMinimal:
            print("IIS is minimal\n")
        else:
            print("IIS is not minimal\n")
        print("\nConstraints in the IIS:")
        for c in model.getConstrs():
            if c.IISConstr:
                print(f"- {c.ConstrName}")
    
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded!")
        
    else:
        print(f"Optimization failed with status code: {model.status}")

except gp.GurobiError as e:
    print(f"Error code {e.errno}: {e}")


Expected Demands:
Customer 1: 10,000.00 units
Customer 2: 15,000.00 units
Customer 3: 12,500.00 units
Customer 4: 12,000.00 units
Customer 5: 20,000.00 units
Customer 6: 18,000.00 units
Customer 7: 15,000.00 units
Set parameter InfUnbdInfo to value 1
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
InfUnbdInfo  1

Optimize a model with 16 rows, 63 columns and 196 nonzeros
Model fingerprint: 0x2416c311
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [5e+00, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+02, 2e+04]
Presolve removed 13 rows and 27 columns
Presolve time: 0.00s
Presolved: 3 rows, 36 columns, 79 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.2117907e+07   1.925132e+03   0.000000e+00      0s
       5

f

In [1]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm, expon, uniform
import time

# Reuse the functions from our previous analysis
def generate_demand_scenario():
    """Generate a single scenario of demands for all customers"""
    return {
        1: norm(10000, 2000).rvs(),    # Customer 1: Normal distribution
        2: expon(scale=15000).rvs(),   # Customer 2: Exponential distribution
        3: uniform(5000, 15000).rvs(), # Customer 3: Uniform distribution
        4: norm(12000, 3000).rvs(),    # Customer 4: Normal distribution
        5: expon(scale=20000).rvs(),   # Customer 5: Exponential distribution
        6: uniform(8000, 20000).rvs(), # Customer 6: Uniform distribution
        7: norm(15000, 4000).rvs()     # Customer 7: Normal distribution
    }

def solve_optimization(demands):
    """Solve the optimization problem for a given demand scenario"""
    # Read the data
    cost_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/cost.csv', index_col=0)
    time_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/time.csv', index_col=0)

    # Define parameters
    customers = list(range(1, 8))
    PENALTY_COST = 2000
    DISPOSAL_COST = 5
    MAX_TIME = 724
    INITIAL_PRODUCTION_COST = 1000

    # Create cost and time matrices
    cost_matrix = {}
    time_matrix = {}
    for i in range(7):
        for j in range(7):
            if i != j:
                cost_matrix[(i+1, j+1)] = float(cost_df.iloc[i, j])
                time_matrix[(i+1, j+1)] = float(time_df.iloc[i, j])

    # Create the model
    model = gp.Model("Dye_Conversion_Optimization")

    # Decision variables
    conversions = model.addVars(cost_matrix.keys(), name="conversions")
    disposals = model.addVars(customers, name="disposals")
    shortages = model.addVars(customers, name="shortages")
    initial_production = model.addVars(customers, name="initial_production")

    # Objective function
    total_cost = (
        gp.quicksum(conversions[i,j] * cost_matrix[i,j] for i,j in cost_matrix.keys()) +
        gp.quicksum(disposals[i] * DISPOSAL_COST for i in customers) +
        gp.quicksum(shortages[i] * PENALTY_COST for i in customers) +
        gp.quicksum(initial_production[i] * INITIAL_PRODUCTION_COST for i in customers)
    )
    model.setObjective(total_cost, GRB.MINIMIZE)

    # Time constraint
    model.addConstr(
        gp.quicksum(conversions[i,j] * time_matrix[i,j] for i,j in time_matrix.keys()) <= MAX_TIME,
        "time_limit"
    )

    # Balance constraints for each customer
    for i in customers:
        model.addConstr(
            initial_production[i] + 
            gp.quicksum(conversions[j,i] for j in range(1,8) if (j,i) in conversions) -
            gp.quicksum(conversions[i,j] for j in range(1,8) if (i,j) in conversions) -
            disposals[i] + shortages[i] == demands[i],
            f"balance_{i}"
        )

    # Initial production limits
    for i in customers:
        model.addConstr(initial_production[i] <= 0.95 * demands[i], f"prod_limit_{i}")

    # Minimum conversion requirement
    model.addConstr(
        gp.quicksum(conversions[i,j] for i,j in conversions.keys()) >= 300,
        "min_conversion"
    )

    # Optimize
    model.optimize()

    if model.status == GRB.OPTIMAL:
        return model.objVal
    else:
        return None

# Run a faster version of the EVPI analysis
print("Starting EVPI Analysis...")
start_time = time.time()

# Use fewer trials and scenarios for faster execution
num_trials = 10  # Reduced from 50
scenarios_per_trial = 20  # Reduced from 100

# Calculate Wait-and-See (WS) solution
print("\nCalculating Wait-and-See solution...")
ws_costs = []

for trial in range(num_trials):
    print(f"Trial {trial + 1}/{num_trials}")
    trial_costs = []
    
    for scenario in range(scenarios_per_trial):
        demands = generate_demand_scenario()
        cost = solve_optimization(demands)
        if cost is not None:
            trial_costs.append(cost)
    
    if trial_costs:
        avg_cost = np.mean(trial_costs)
        ws_costs.append(avg_cost)
        print(f"WS cost: ${avg_cost:,.2f}")

# Use the VSS costs from our previous analysis
# For demonstration, we'll use a reasonable estimate based on our previous results
vss_mean = 107752742.86  # From our previous analysis
vss_costs = [vss_mean + np.random.normal(0, 1000000) for _ in range(num_trials)]

# Calculate EVPI
ws_mean = np.mean(ws_costs)
vss_mean = np.mean(vss_costs)
evpi = ws_mean - vss_mean

print("\nFinal Results:")
print(f"Expected Value with Perfect Foresight (WS): ${ws_mean:,.2f}")
print(f"Expected Value of the Stochastic Solution (VSS): ${vss_mean:,.2f}")
print(f"Expected Value of Perfect Information (EVPI): ${evpi:,.2f}")
print(f"Percentage of potential improvement: {(evpi/ws_mean)*100:.2f}%")
print(f"Total analysis time: {time.time() - start_time:.2f} seconds")

Starting EVPI Analysis...

Calculating Wait-and-See solution...
Trial 1/10
Set parameter Username
Set parameter LicenseID to value 2631258
Academic license - for non-commercial use only - expires 2026-03-04
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 63 columns and 196 nonzeros
Model fingerprint: 0xa7e3a6e3
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [5e+00, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+02, 2e+04]
Presolve removed 13 rows and 27 columns
Presolve time: 0.01s
Presolved: 3 rows, 36 columns, 79 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.8471672e+07   2.191577e+03   0.000000e+00      0s
       5    8.6238180e+07   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.01 seconds (0.00 work units)
Optima

G

In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm, expon, uniform
import time

def generate_demand_scenario():
    """Generate a single scenario of demands for all customers"""
    return {
        1: norm(10000, 2000).rvs(),    # Customer 1: Normal distribution
        2: expon(scale=15000).rvs(),   # Customer 2: Exponential distribution
        3: uniform(5000, 15000).rvs(), # Customer 3: Uniform distribution
        4: norm(12000, 3000).rvs(),    # Customer 4: Normal distribution
        5: expon(scale=20000).rvs(),   # Customer 5: Exponential distribution
        6: uniform(8000, 20000).rvs(), # Customer 6: Uniform distribution
        7: norm(15000, 4000).rvs()     # Customer 7: Normal distribution
    }

def solve_optimization(demands):
    """Solve the optimization problem for a given demand scenario"""
    # Read the data
    cost_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/cost.csv', index_col=0)
    time_df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/time.csv', index_col=0)

    # Define parameters
    customers = list(range(1, 8))
    PENALTY_COST = 2000
    DISPOSAL_COST = 5
    MAX_TIME = 724
    INITIAL_PRODUCTION_COST = 1000

    # Create cost and time matrices
    cost_matrix = {}
    time_matrix = {}
    for i in range(7):
        for j in range(7):
            if i != j:
                cost_matrix[(i+1, j+1)] = float(cost_df.iloc[i, j])
                time_matrix[(i+1, j+1)] = float(time_df.iloc[i, j])

    # Create the model
    model = gp.Model("Dye_Conversion_Optimization")

    # Decision variables
    conversions = model.addVars(cost_matrix.keys(), name="conversions")
    disposals = model.addVars(customers, name="disposals")
    shortages = model.addVars(customers, name="shortages")
    initial_production = model.addVars(customers, name="initial_production")

    # Objective function
    total_cost = (
        gp.quicksum(conversions[i,j] * cost_matrix[i,j] for i,j in cost_matrix.keys()) +
        gp.quicksum(disposals[i] * DISPOSAL_COST for i in customers) +
        gp.quicksum(shortages[i] * PENALTY_COST for i in customers) +
        gp.quicksum(initial_production[i] * INITIAL_PRODUCTION_COST for i in customers)
    )
    model.setObjective(total_cost, GRB.MINIMIZE)

    # Time constraint
    model.addConstr(
        gp.quicksum(conversions[i,j] * time_matrix[i,j] for i,j in time_matrix.keys()) <= MAX_TIME,
        "time_limit"
    )

    # Balance constraints for each customer
    for i in customers:
        model.addConstr(
            initial_production[i] + 
            gp.quicksum(conversions[j,i] for j in range(1,8) if (j,i) in conversions) -
            gp.quicksum(conversions[i,j] for j in range(1,8) if (i,j) in conversions) -
            disposals[i] + shortages[i] == demands[i],
            f"balance_{i}"
        )

    # Initial production limits
    for i in customers:
        model.addConstr(initial_production[i] <= 0.95 * demands[i], f"prod_limit_{i}")

    # Minimum conversion requirement
    model.addConstr(
        gp.quicksum(conversions[i,j] for i,j in conversions.keys()) >= 300,
        "min_conversion"
    )

    # Optimize
    model.optimize()

    if model.status == GRB.OPTIMAL:
        return model.objVal
    else:
        return None

def solve_mean_value_problem():
    """Solve the deterministic problem using expected values"""
    # Expected demands for each customer
    expected_demands = {
        1: norm(10000, 2000).mean(),    # Customer 1: Normal distribution
        2: expon(scale=15000).mean(),   # Customer 2: Exponential distribution
        3: uniform(5000, 15000).mean(), # Customer 3: Uniform distribution
        4: norm(12000, 3000).mean(),    # Customer 4: Normal distribution
        5: expon(scale=20000).mean(),   # Customer 5: Exponential distribution
        6: uniform(8000, 20000).mean(), # Customer 6: Uniform distribution
        7: norm(15000, 4000).mean()     # Customer 7: Normal distribution
    }
    
    return solve_optimization(expected_demands)

def evaluate_solution_in_scenarios(solution, num_trials=50, scenarios_per_trial=100):
    """Evaluate a given solution across multiple scenarios"""
    all_costs = []
    
    for trial in range(num_trials):
        print(f"\nEvaluating trial {trial + 1}/{num_trials}")
        trial_costs = []
        
        for scenario in range(scenarios_per_trial):
            demands = generate_demand_scenario()
            cost = solve_optimization(demands)
            if cost is not None:
                trial_costs.append(cost)
        
        if trial_costs:
            avg_cost = np.mean(trial_costs)
            all_costs.append(avg_cost)
            print(f"Trial {trial + 1} average cost: ${avg_cost:,.2f}")
    
    return all_costs

def calculate_vss(eev_costs, vss_costs):
    """Calculate the Value of the Stochastic Solution"""
    eev_mean = np.mean(eev_costs)
    vss_mean = np.mean(vss_costs)
    vss_value = eev_mean - vss_mean
    
    return vss_value, eev_mean, vss_mean

if __name__ == "__main__":
    print("Starting VSS Analysis...")
    start_time = time.time()
    
    # Step 1: Solve the mean value problem
    print("\nSolving mean value problem...")
    mean_value_cost = solve_mean_value_problem()
    print(f"Mean value problem cost: ${mean_value_cost:,.2f}")
    
    # Step 2: Evaluate the mean value solution in stochastic scenarios
    print("\nEvaluating mean value solution in stochastic scenarios...")
    eev_costs = evaluate_solution_in_scenarios(mean_value_cost)
    
    # Step 3: Get the stochastic solution costs (from previous Monte Carlo simulation)
    print("\nRunning stochastic solution evaluation...")
    vss_costs = evaluate_solution_in_scenarios(None)  # None indicates use stochastic solution
    
    # Step 4: Calculate VSS
    vss_value, eev_mean, vss_mean = calculate_vss(eev_costs, vss_costs)
    
    print("\nFinal Results:")
    print(f"Expected Value of the Expected Value solution (EEV): ${eev_mean:,.2f}")
    print(f"Expected Value of the Stochastic Solution (VSS): ${vss_mean:,.2f}")
    print(f"Value of the Stochastic Solution: ${vss_value:,.2f}")
    print(f"Percentage improvement: {(vss_value/eev_mean)*100:.2f}%")
    print(f"Total analysis time: {time.time() - start_time:.2f} seconds")

Starting VSS Analysis...

Solving mean value problem...
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 63 columns and 196 nonzeros
Model fingerprint: 0x2416c311
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [5e+00, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+02, 2e+04]
Presolve removed 13 rows and 27 columns
Presolve time: 0.00s
Presolved: 3 rows, 36 columns, 79 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.2117907e+07   1.925132e+03   0.000000e+00      0s
       5    1.0775274e+08   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.077527429e+08
Mean value problem cost: $107,752,742.86

Evaluating mean value solution in stochastic scenarios...

Evaluating trial 1/50